In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import joblib
from sklearn.ensemble import VotingClassifier

In [2]:
cv_scores = {}

with open('../cv_scores.txt', 'r') as f:
    for i in f.readlines():
        cv_scores[i.split(': ')[0]] = float(i.split(': ')[1].rstrip())

for model_name, score in cv_scores.items():
    print(f'{model_name}: {score}')

Logistic Regression: 0.9583398762857318
SVM Poly: 0.9488357904987282
SVM RBF: 0.9592979227799867
SVM Linear: 0.9542525100059439
KNN: 0.832491126270462
Random Forest: 0.9507921780204096
Ridge: 0.9560088981117433


In [3]:
cv_scores_df = pd.DataFrame({
    "Model": np.array([i for i in cv_scores.keys()]),
    "CV Score": np.array([i for i in cv_scores.values()])
})

cv_scores_df

,Model,CV Score
0,Logistic Regression,0.958340
1,SVM Poly,0.948836
2,SVM RBF,0.959298
3,SVM Linear,0.954253
4,KNN,0.832491
5,Random Forest,0.950792
6,Ridge,0.956009


In [4]:
models_df = pd.read_csv('../reports/model_report.csv')

models_df

,Model,Accuracy,Precision,Recall,F1,Loss (Log Loss)
0,Random Forest,0.9922,0.9889,0.9963,0.9926,0.0448
1,SVM RBF,0.9848,0.9804,0.9908,0.9856,0.0488
2,KNN,0.9835,0.9755,0.9936,0.9845,0.3066
3,SVM Poly,0.9822,0.9736,0.9930,0.9832,0.0507
4,Logistic Regression,0.9693,0.9684,0.9734,0.9709,0.0805
5,SVM Linear,0.9683,0.9633,0.9768,0.9700,0.0893
6,Ridge Classifier,0.9642,0.9631,0.9690,0.9660,0.0963


In [5]:
merged_df = pd.merge(models_df, cv_scores_df, on='Model')

merged_df

,Model,Accuracy,Precision,Recall,F1,Loss (Log Loss),CV Score
0,Random Forest,0.9922,0.9889,0.9963,0.9926,0.0448,0.950792
1,SVM RBF,0.9848,0.9804,0.9908,0.9856,0.0488,0.959298
2,KNN,0.9835,0.9755,0.9936,0.9845,0.3066,0.832491
3,SVM Poly,0.9822,0.9736,0.9930,0.9832,0.0507,0.948836
4,Logistic Regression,0.9693,0.9684,0.9734,0.9709,0.0805,0.958340
5,SVM Linear,0.9683,0.9633,0.9768,0.9700,0.0893,0.954253


In [6]:
eval_df = merged_df[['Model', 'F1', 'CV Score', 'Loss (Log Loss)']]

eval_df

,Model,F1,CV Score,Loss (Log Loss)
0,Random Forest,0.9926,0.950792,0.0448
1,SVM RBF,0.9856,0.959298,0.0488
2,KNN,0.9845,0.832491,0.3066
3,SVM Poly,0.9832,0.948836,0.0507
4,Logistic Regression,0.9709,0.958340,0.0805
5,SVM Linear,0.9700,0.954253,0.0893


In [7]:
eval_df['Ranking'] = eval_df['F1'] + eval_df['CV Score'] + eval_df['Loss (Log Loss)']

eval_df = eval_df.sort_values(by='Ranking', ascending=False)

eval_df

,Model,F1,CV Score,Loss (Log Loss),Ranking
2,KNN,0.9845,0.832491,0.3066,2.123591
5,SVM Linear,0.9700,0.954253,0.0893,2.013553
4,Logistic Regression,0.9709,0.958340,0.0805,2.009740
1,SVM RBF,0.9856,0.959298,0.0488,1.993698
0,Random Forest,0.9926,0.950792,0.0448,1.988192
3,SVM Poly,0.9832,0.948836,0.0507,1.982736


In [8]:
print('Loading vectors:')
x_tr = joblib.load('../vectors/x_training_vector.pkl')
y_tr = joblib.load('../vectors/y_training_vector.pkl')
print('x_tr shape:', x_tr.shape)
print('y_tr shape:', y_tr.shape)

Loading vectors:
x_tr shape: (326222, 5000)
y_tr shape: (326222,)


In [9]:
print('Running feature selection:')
sel = joblib.load('../models/sel.pkl')
x_tr_s = sel.fit_transform(x_tr, y_tr)

Running feature selection:


In [10]:
print('Loading models:', flush=True)

models = []

model_names = ['Random Forest', 'SVM RBF', 'SVM Poly']

for name in tqdm(model_names, total=len(model_names), desc='Loading models'):
    print(f'Loading: {name}', flush=True)
    models.append((name, joblib.load(f'../models/model_{name}.pkl')))
    print(f'Loaded: {name}')

print(models)

Loading models:


Loading models:   0%|                                                                            | 0/3 [00:00<?, ?it/s]

Loading: Random Forest


Loading models:  33%|██████████████████████▋                                             | 1/3 [00:01<00:03,  1.95s/it]

Loaded: Random Forest
Loading: SVM RBF


Loading models:  67%|█████████████████████████████████████████████▎                      | 2/3 [00:02<00:00,  1.14it/s]

Loaded: SVM RBF
Loading: SVM Poly


Loading models: 100%|████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.35it/s]

Loaded: SVM Poly
[('Random Forest', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, verbose=2)), ('SVM RBF', SVC(C=1, class_weight='balanced', probability=True, random_state=42)), ('SVM Poly', SVC(C=1, class_weight='balanced', degree=2, kernel='poly', probability=True,
    random_state=42))]


In [12]:
print('Building ensemble:')

ensemble = VotingClassifier(
    estimators=models,
    voting='soft',
    weights=[2, 1, 1]
)

ensemble.fit(x_tr_s, y_tr)

print('Ensemble training complete.')

Building ensemble:


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


building tree 7 of 200building tree 8 of 200
building tree 4 of 200
building tree 6 of 200
building tree 5 of 200
building tree 2 of 200
building tree 3 of 200
building tree 11 of 200
building tree 12 of 200
building tree 10 of 200
building tree 1 of 200

building tree 9 of 200
building tree 13 of 200
building tree 14 of 200
building tree 15 of 200
building tree 16 of 200
building tree 17 of 200
building tree 18 of 200
building tree 19 of 200
building tree 20 of 200
building tree 21 of 200
building tree 22 of 200
building tree 23 of 200
building tree 24 of 200
building tree 25 of 200
building tree 26 of 200
building tree 27 of 200
building tree 28 of 200
building tree 29 of 200


[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  2.5min


building tree 30 of 200
building tree 31 of 200
building tree 32 of 200
building tree 33 of 200
building tree 34 of 200
building tree 35 of 200
building tree 36 of 200
building tree 37 of 200
building tree 38 of 200
building tree 39 of 200
building tree 40 of 200
building tree 41 of 200
building tree 42 of 200
building tree 43 of 200
building tree 44 of 200
building tree 45 of 200
building tree 46 of 200
building tree 47 of 200
building tree 48 of 200
building tree 49 of 200
building tree 50 of 200
building tree 51 of 200
building tree 52 of 200
building tree 53 of 200
building tree 54 of 200
building tree 55 of 200building tree 56 of 200

building tree 57 of 200
building tree 58 of 200
building tree 59 of 200
building tree 60 of 200
building tree 61 of 200
building tree 62 of 200
building tree 63 of 200
building tree 64 of 200
building tree 65 of 200
building tree 66 of 200
building tree 67 of 200
building tree 68 of 200
building tree 69 of 200
building tree 70 of 200
building tree 71

[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed: 15.4min


building tree 151 of 200
building tree 152 of 200
building tree 153 of 200
building tree 154 of 200
building tree 155 of 200
building tree 156 of 200
building tree 157 of 200
building tree 158 of 200
building tree 159 of 200
building tree 160 of 200
building tree 161 of 200
building tree 162 of 200
building tree 163 of 200
building tree 164 of 200
building tree 165 of 200
building tree 166 of 200
building tree 167 of 200
building tree 168 of 200
building tree 169 of 200
building tree 170 of 200
building tree 171 of 200
building tree 172 of 200
building tree 173 of 200
building tree 174 of 200
building tree 175 of 200
building tree 176 of 200
building tree 177 of 200
building tree 178 of 200
building tree 179 of 200
building tree 180 of 200
building tree 181 of 200
building tree 182 of 200
building tree 183 of 200
building tree 184 of 200
building tree 185 of 200
building tree 186 of 200
building tree 187 of 200
building tree 188 of 200
building tree 189 of 200
building tree 190 of 200


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 21.3min finished
C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(
C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(


Ensemble training complete.


In [14]:
joblib.dump(ensemble, f'../models/ensemble_model.pkl')
print('Ensemble saved')

Ensemble saved


In [15]:
!jupyter nbconvert --to script ensemble_build.ipynb

[NbConvertApp] Converting notebook ensemble_build.ipynb to script
[NbConvertApp] Writing 2111 bytes to ensemble_build.py
